In [1]:
using Lux, LuxCUDA, MLUtils
using Optimisers, Random, Statistics
using Zygote
using DiffEqFlux, OrdinaryDiffEq
using FFTW
#using Reactant
using Images, JLD2
using ComponentArrays
using DeepEquilibriumNetworks
using Plots
using Dates
using SciMLSensitivity, Lux, NonlinearSolve, LinearSolve

In [2]:
#Reactant.set_default_backend("cpu")
#const device = reactant_device()
const cdev = cpu_device()
const gdev = gpu_device()
dev = gdev

(::CUDADevice{Nothing, Missing}) (generic function with 1 method)

In [3]:
model_first_conv = Chain(
        
        Conv((5, 5), 1 => 8, tanh; pad = 2),
        Conv((5, 5), 8 => 16; pad = 2),
        SkipConnection(Chain(
            Conv((1, 1), 16 => 64, gelu),
            Conv((1, 1), 64 => 32, gelu),
            Conv((1, 1), 32 => 16),
        ), +),
    ) 
model_conv = Chain(
        Parallel(+, 
            NoOpLayer(), # Pass z through
            NoOpLayer()  # Pass x through
        ),
        model_first_conv,
        GroupNorm(16, 4),
        Conv((5, 5), 16 => 32, tanh; pad = 2),
        Conv((5, 5), 32 => 64; pad = 2),
        SkipConnection(Chain(
            Conv((1, 1), 64 => 128, gelu),
            Conv((1, 1), 128 => 128, gelu),
            Conv((1, 1), 128 => 64, gelu),
        ), +),
        GroupNorm(64, 4),
        Conv((1, 1), 64 => 64, gelu),
        Conv((1, 1), 64 => 32, gelu),
        Conv((1, 1), 32 => 1),
    ) 


Chain(
    layer_1 = Parallel(
        connection = +,
        layer_(1-2) = NoOpLayer(),
    ),
    layer_2 = Chain(
        layer_1 = Conv((5, 5), 1 => 8, tanh, pad=2),  # 208 parameters
        layer_2 = Conv((5, 5), 8 => 16, pad=2),   # 3_216 parameters
        layer_3 = SkipConnection(
            connection = +,
            layers = Chain(
                layer_1 = Conv((1, 1), 16 => 64, gelu_tanh),  # 1_088 parameters
                layer_2 = Conv((1, 1), 64 => 32, gelu_tanh),  # 2_080 parameters
                layer_3 = Conv((1, 1), 32 => 16),  # 528 parameters
            ),
        ),
    ),
    layer_3 = GroupNorm(16, 4, affine=true),      # 32 parameters
    layer_4 = Conv((5, 5), 16 => 32, tanh, pad=2),  # 12_832 parameters
    layer_5 = Conv((5, 5), 32 => 64, pad=2),      # 51_264 parameters
    layer_6 = SkipConnection(
        connection = +,
        layers = Chain(
            layer_1 = Conv((1, 1), 64 => 128, gelu_tanh),  # 8_320 parameters
            layer_2 = Con

In [4]:
model = SkipConnection(connection = +, layers = DeepEquilibriumNetwork(model_conv, NewtonRaphson(; linsolve=KrylovJL_GMRES()); init = nothing, verbose=false, linsolve_kwargs=(; maxiters=10), maxiters=10))

SkipConnection(
    connection = +,
    layers = DeepEquilibriumNetwork(
        model = Chain(
            layer_1 = Parallel(
                connection = +,
                layer_(1-2) = NoOpLayer(),
            ),
            layer_2 = Chain(
                layer_1 = Conv((5, 5), 1 => 8, tanh, pad=2),  # 208 parameters
                layer_2 = Conv((5, 5), 8 => 16, pad=2),  # 3_216 parameters
                layer_3 = SkipConnection(
                    connection = +,
                    layers = Chain(
                        layer_1 = Conv((1, 1), 16 => 64, gelu_tanh),  # 1_088 parameters
                        layer_2 = Conv((1, 1), 64 => 32, gelu_tanh),  # 2_080 parameters
                        layer_3 = Conv((1, 1), 32 => 16),  # 528 parameters
                    ),
                ),
            ),
            layer_3 = GroupNorm(16, 4, affine=true),  # 32 parameters
            layer_4 = Conv((5, 5), 16 => 32, tanh, pad=2),  # 12_832 parameters
            layer_5 = C

In [4]:
model = DeepEquilibriumNetwork(model_conv, Tsit5(); init = nothing,save_everystep = false, reltol = 1e-3, abstol = 1e-4, save_start = false)

DeepEquilibriumNetwork(
    model = Chain(
        layer_1 = Parallel(
            connection = +,
            layer_(1-2) = NoOpLayer(),
        ),
        layer_2 = Chain(
            layer_1 = Conv((5, 5), 1 => 8, tanh, pad=2),  # 208 parameters
            layer_2 = Conv((5, 5), 8 => 16, pad=2),  # 3_216 parameters
            layer_3 = SkipConnection(
                connection = +,
                layers = Chain(
                    layer_1 = Conv((1, 1), 16 => 64, gelu_tanh),  # 1_088 parameters
                    layer_2 = Conv((1, 1), 64 => 32, gelu_tanh),  # 2_080 parameters
                    layer_3 = Conv((1, 1), 32 => 16),  # 528 parameters
                ),
            ),
        ),
        layer_3 = GroupNorm(16, 4, affine=true),  # 32 parameters
        layer_4 = Conv((5, 5), 16 => 32, tanh, pad=2),  # 12_832 parameters
        layer_5 = Conv((5, 5), 32 => 64, pad=2),  # 51_264 parameters
        layer_6 = SkipConnection(
            connection = +,
            laye

In [5]:
rng = Xoshiro()
ps , st = Lux.setup(rng, model)

((model = (layer_1 = (layer_1 = NamedTuple(), layer_2 = NamedTuple()), layer_2 = (layer_1 = (weight = Float32[0.50820893 -0.5344653 … -0.35562846 -0.27828345; -0.54660004 -0.07102222 … -0.41381437 -0.2615839; … ; -0.35558316 0.06278181 … 0.57380795 -0.21849534; 0.22155622 0.33716187 … -0.35255986 0.32573658;;;; -0.028913599 0.4637741 … 0.57422763 0.39607295; -0.38936466 0.4443873 … -0.42404047 -0.25743502; … ; -0.49028695 0.41997513 … 0.16189599 0.27257562; -0.5207961 0.14485927 … -0.08794228 0.43137574;;;; 0.23538354 0.4048713 … 0.36735144 -0.42050332; -0.025929049 0.21072803 … -0.42825893 0.3895778; … ; -0.48638186 -0.44939613 … -0.43426868 0.3719449; -0.10571682 -0.47356284 … -0.34728584 -0.43170583;;;; -0.006118451 -0.049241353 … 0.177148 0.37479022; -0.51201403 0.03355292 … 0.5280842 0.574539; … ; -0.095253825 0.022326447 … 0.09058766 0.5300609; 0.065796845 -0.25001177 … -0.1877759 0.0014969549;;;; -0.2872112 -0.37881404 … 0.37692645 -0.34071803; -0.16662195 -0.43979585 … -0.47348

In [18]:
ps_cpu = ps
@load "ps_latest.jld2" ps
ps_old = ps |> cdev
ps = ps_cpu

(model = (layer_1 = (layer_1 = NamedTuple(), layer_2 = NamedTuple()), layer_2 = (layer_1 = (weight = Float32[0.12792103 0.500686 … 0.22813566 -0.08005894; -0.24574082 0.160927 … 0.24972656 -0.07666701; … ; 0.38388285 0.5735064 … 0.4730502 -0.5652282; -0.29599458 -0.10223514 … 0.24102014 0.41761264;;;; -0.00070587447 -0.32385033 … -0.23155373 -0.49744922; -0.5459249 0.4230145 … -0.34088072 0.48150647; … ; -0.008996114 0.55061024 … -0.57051325 -0.5567517; 0.53024554 -0.39639834 … -0.43282557 0.026424248;;;; 0.27838475 -0.5186128 … -0.076143526 0.5282329; 0.2768315 0.503302 … 0.17910925 -0.028895842; … ; 0.16840537 0.27975217 … -0.31563717 -0.18863676; -0.4154235 0.024811942 … -0.0149719585 -0.052190594;;;; -0.044407804 0.32494974 … -0.16273455 0.1993445; -0.36271712 -0.5504388 … -0.10069159 0.4659075; … ; -0.36855114 -0.40712956 … 0.26750997 -0.23597151; 0.19768491 0.37092587 … 0.08872923 0.13683476;;;; -0.14419867 0.18940182 … -0.20831838 0.34516504; 0.4758978 -0.41832656 … -0.47462288 

In [21]:
ps_old

ComponentVector{Float32}(layer_1 = (layer_1 = (weight = Float32[-0.5156039 0.31830207 … 0.54628533 0.20379029; -0.054568622 -0.45326352 … 0.42619815 -0.46810475; … ; 0.052183174 -0.10723932 … 0.12594105 -0.31320214; 0.20721574 -0.54464287 … 0.19336234 0.31061116;;;; 0.14788787 -0.005361416 … -0.32622296 0.032514222; -0.18747962 -0.036960877 … 0.20209257 0.25226864; … ; 0.36388606 -0.45006093 … 0.38368112 0.102261476; 0.22638045 -0.045780875 … 0.07437828 -0.099924296;;;; 0.42248276 0.014029021 … -0.112267114 -0.22378059; -0.1426463 -0.3524065 … -0.057274412 -0.54889005; … ; -0.3172466 0.36811435 … -0.4447652 0.5174231; 0.5102624 0.17435746 … -0.43966433 0.14375825;;;; 0.11226511 -0.20781566 … 0.295928 -0.14229618; -0.22354339 -0.10709284 … -0.30813083 -0.55016106; … ; -0.36722964 0.27480987 … 0.30162057 0.3756912; -0.27520102 0.28823578 … 0.51057607 -0.45854425;;;; -0.20631212 0.4666755 … 0.2519986 -0.008068724; 0.37324235 0.14082547 … 0.35090733 0.020776508; … ; -0.46206975 -0.22304013

In [6]:
ps = ps |> ComponentArray |> dev
st = st |> dev

opt = Optimisers.NAdam(1e-4)
state = Optimisers.setup(opt,ps)
train_state = Lux.Training.TrainState(model, ps, st, opt)

TrainState(
    SkipConnection(
        connection = +,
        layers = DeepEquilibriumNetwork(
            model = Chain(
                layer_1 = Parallel(
                    connection = +,
                    layer_(1-2) = NoOpLayer(),
                ),
                layer_2 = Chain(
                    layer_1 = Conv((5, 5), 1 => 8, tanh, pad=2),  # 208 parameters
                    layer_2 = Conv((5, 5), 8 => 16, pad=2),  # 3_216 parameters
                    layer_3 = SkipConnection(
                        connection = +,
                        layers = Chain(
                            layer_1 = Conv((1, 1), 16 => 64, gelu_tanh),  # 1_088 parameters
                            layer_2 = Conv((1, 1), 64 => 32, gelu_tanh),  # 2_080 parameters
                            layer_3 = Conv((1, 1), 32 => 16),  # 528 parameters
                        ),
                    ),
                ),
                layer_3 = GroupNorm(16, 4, affine=true),  # 32 parameters
       

In [7]:
function loss_function(model, ps, st, (x, y_true))
    y, st = model(x, ps, st)
    y_pred = y
    #y_pred = model(x, ps, st)[1][1]
    loss_mse= MSELoss()
    mse_loss = loss_mse(y_pred, y_true)
    #sptrl_loss = loss_mse(dct(dct(y_pred,1),2),dct(dct(y_pred,1),2))
    return mse_loss, st
    #return mes_loss + sptrl_loss, st
end

loss_function (generic function with 1 method)

In [8]:
x = randn(Float32,128,128,1,4)
y_true = randn(Float32,128,128,1,4)
x_dev = x |> dev
y_dev = y_true |> dev
y, _ = model(x_dev, ps, st)

(Float32[-0.07479806 -2.0140436 … -0.15445139 -0.7268449; -0.01568681 -1.2662598 … 1.3152964 -0.23717938; … ; -1.3061903 -1.3260565 … -0.37143552 0.89892894; 0.39161062 -0.7245754 … 0.9460227 0.062338393;;;; -0.28583398 0.9968238 … 0.84769297 -0.74867165; 0.11165659 1.123834 … -0.53996503 -0.31907362; … ; 0.9644849 0.2705218 … 1.1373845 1.4038311; -0.69183105 -1.5327475 … 0.3284298 -0.6827809;;;; -0.64805037 -0.7347776 … 0.70058477 0.48413405; -1.027164 0.22337985 … 0.7028477 0.894149; … ; 0.71550184 0.7471838 … 0.68629587 1.3945422; -1.2528013 0.75479186 … -0.34661156 0.13048625;;;; -2.7376502 -2.5471277 … 0.32395172 -0.00507332; 0.35320008 0.97387034 … 0.2819054 1.0537599; … ; 1.8910655 1.7695032 … -1.5410701 0.71855533; 0.65196997 -2.2226815 … -0.8854214 -0.5078094], (model = (layer_1 = (layer_1 = NamedTuple(), layer_2 = NamedTuple()), layer_2 = (layer_1 = NamedTuple(), layer_2 = NamedTuple(), layer_3 = (layer_1 = NamedTuple(), layer_2 = NamedTuple(), layer_3 = NamedTuple())), layer

In [9]:
loss_function(model,ps,st,(x_dev,y_dev))

(2.106367f0, (model = (layer_1 = (layer_1 = NamedTuple(), layer_2 = NamedTuple()), layer_2 = (layer_1 = NamedTuple(), layer_2 = NamedTuple(), layer_3 = (layer_1 = NamedTuple(), layer_2 = NamedTuple(), layer_3 = NamedTuple())), layer_3 = NamedTuple(), layer_4 = NamedTuple(), layer_5 = NamedTuple(), layer_6 = (layer_1 = NamedTuple(), layer_2 = NamedTuple(), layer_3 = NamedTuple()), layer_7 = NamedTuple(), layer_8 = NamedTuple(), layer_9 = NamedTuple(), layer_10 = NamedTuple()), fixed_depth = Val{0}(), init = NamedTuple(), solution = DeepEquilibriumSolution
 * Initial Guess: Float32[0.0766015 0.258459 … -0.0136271 -0.0179842; -0.184068 -0.155279 … -0.0921291 -0.163967; … ; -0.121754 -0.0969039 … -0.0332756 -0.16811; -0.10729 -0.267949 … -0.0240485 -0.0667093;;;; -0.121607 0.11329 … 0.024055 0.0847088; 0.291013 -0.0358399 … -0.0792859 -0.142642; … ; 0.137369 0.111235 … 0.297565 0.00156604; -0.145648 -0.192391 … -0.00905567 -0.113125;;;; 0.00161693 -0.0290888 … 0.08907 -0.0177574; 0.240766 

In [ ]:
Training.single_train_step!(AutoZygote(), loss_function, (x_dev, y_dev), train_state)

CUDA.OutOfGPUMemoryError: Out of GPU memory trying to allocate 64.000 MiB
Effective GPU memory usage: 99.68% (9.615 GiB/9.646 GiB)
Memory pool usage: 9.299 GiB (9.344 GiB reserved)


In [11]:
#(loss, st), back = Zygote.pullback(ps -> loss_function(model, ps, st, (x_dev, y_dev)), ps)

In [12]:
#loss

In [13]:
#grads = back((one(loss), nothing))